# DOH 골프 3D 회전 — NLF 버전 (최신 Colab 대응)

mmpose는 최신 Colab(py3.12·torch2.11)에서 사망 → **NLF**(TorchScript, 토치버전 안 탐)로 교체.

**하는 법:** 위에서부터 회색칸 **▶** 순서대로.
**먼저:** 런타임 → 런타임 유형 변경 → **T4 GPU** → 저장.

> NLF는 비상업 연구용 라이선스 — 지금은 '검증'이라 OK. 서비스화 땐 별도 검토.


### 1칸. 모델 준비 (1~2분)
끝에 **`>>> NLF OK`** 뜨면 성공.


In [ ]:
import torch, torchvision, os
import torchvision.ops                      # NLF 모델이 torchvision::nms 를 써서 등록 필요
print('torch', torch.__version__, '| tv', torchvision.__version__, '| cuda', torch.version.cuda)
URL='https://github.com/isarandi/nlf/releases/download/v0.3.2/nlf_l_multi_0.3.2.torchscript'
M='nlf_l_multi_0.3.2.torchscript'
if (not os.path.exists(M)) or os.path.getsize(M) < 10_000_000:
    !wget -q '{URL}' -O '{M}'
sz = os.path.getsize(M)/1e6 if os.path.exists(M) else 0
print('모델 MB:', round(sz,1))
if sz < 10:
    print('❌ 다운로드 실패 — 캡처해서 알려주세요')
else:
    model = torch.jit.load(M).cuda().eval()
    print('>>> NLF OK')


### 2칸. 스윙 영상 올리기


In [ ]:
from google.colab import files
up = files.upload()
VIDEO = list(up.keys())[0]
print('올린 영상:', VIDEO)


### 3칸. 3D 분석 (제일 오래 걸림)
첫 프레임에서 결과 구조를 한 번 찍어봐요(디버그). 이상하면 그 출력만 보내주면 바로 고침.


In [ ]:
import cv2, numpy as np, pickle, torch
cap = cv2.VideoCapture(VIDEO)
J = []; dbg = True
with torch.inference_mode():
    while True:
        ok, fr = cap.read()
        if not ok: break
        rgb = cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)
        t = torch.from_numpy(rgb).permute(2,0,1).unsqueeze(0).cuda()   # [1,3,H,W] uint8
        pred = model.detect_smpl_batched(t)
        if dbg:
            print('pred keys:', list(pred.keys()) if hasattr(pred,'keys') else type(pred))
            jj = pred['joints3d']
            print('joints3d:', type(jj), 'len', len(jj) if hasattr(jj,'__len__') else '?')
            dbg = False
        per_img = pred['joints3d'][0]        # 첫(유일) 이미지의 검출들
        if per_img is None or len(per_img) == 0:
            continue
        kp = per_img[0]                       # 첫 사람
        kp = kp.detach().cpu().numpy() if torch.is_tensor(kp) else np.asarray(kp)
        J.append(kp)
cap.release()
J = np.array(J)
pickle.dump({'joints': J}, open('joints.pkl','wb'))
print('저장 완료 · frames', J.shape)


### 4칸. 회전 숫자 뽑기
먼저 `--check`로 관절 수(K) 확인. K가 24면 `--skeleton smpl`, 17이면 `h36m`.
P4(백스윙탑) 프레임은 영상 보고 대략.


In [ ]:
!wget -q https://raw.githubusercontent.com/tinyalex3628-dotcom/doh-golf-survey/claude/ai-video-analysis-engine-wlr06k/pose3d_poc/wham_golf_rotation.py -O rot.py
!python rot.py joints.pkl --check     # ← 관절 수(K) 확인용


In [ ]:
# 탑/임팩트/어드레스 자동검출됨. 특정 프레임 강제하려면 --p4 46 처럼 추가.
!python rot.py joints.pkl --skeleton smpl --png rot.png
from IPython.display import Image; import os
if os.path.exists('rot.png'): display(Image('rot.png'))


### 5칸. 형한테 보낼 것
- 3칸의 **pred keys / joints3d 구조** 출력
- 4칸 `--check`의 **관절 수(shape)**
- **백스윙탑 흉곽 숫자** + **그래프(rot.png)**
- 빨간 에러 있으면 그 화면
